# 02 — Feature Engineering

## Objective

Transform the cleaned EV charging dataset into a feature-rich dataset suitable for exploratory analysis and machine learning.

We will extract useful temporal information from `StartDatetime`, derive a High/Normal Demand target according to the GUVI project requirements, and distinguish predictors from columns that must not be used because of identifier issues or potential data leakage.

The cleaned dataset will remain unchanged. The engineered dataset will be saved separately under `data/processed/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Paths are relative to the notebooks directory.
input_path = Path("../data/processed/ev_charging_cleaned.csv")

if not input_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {input_path.resolve()}"
    )

# Load the output generated by Notebook 01.
df = pd.read_csv(input_path)

# Reconstruct datetime objects from the saved CSV strings.
# errors="raise" stops execution if a timestamp is invalid.
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="raise"
)
df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="raise"
)

print(f"Dataset shape: {df.shape}")
print(f"Start timestamp missing: {df['StartDatetime'].isna().sum()}")
print(f"Demand missing: {df['Demand'].isna().sum()}")

display(df.head())

Dataset shape: (72826, 15)
Start timestamp missing: 0
Demand missing: 0


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,DurationValidated,DurationValidationStatus
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36,185.0,valid
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19,90.0,valid
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78,60.0,valid
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65,344.0,valid
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,2022-09-26 00:30:14,300,33.81,300.0,valid


## Temporal Feature Engineering

The time of a charging session may help explain variations in demand. We will extract the year, month, day of the month, hour, day of the week, weekend status, and a broad time-of-day category.

These features allow us to compare demand across temporal periods and give the regression and classification pipelines information about when a session starts.

In [2]:
# Work from the session's start timestamp.
start_time = df["StartDatetime"]

# Calendar features.
df["StartYear"] = start_time.dt.year
df["StartMonth"] = start_time.dt.month
df["StartDayOfMonth"] = start_time.dt.day

# Time features.
df["StartHour"] = start_time.dt.hour
df["StartDayOfWeek"] = start_time.dt.dayofweek

# Monday-Friday = weekday; Saturday-Sunday = weekend.
df["IsWeekend"] = (
    df["StartDayOfWeek"].isin([5, 6]).astype("int8")
)

# Retain a readable day name for plots and categorical models.
df["StartDayName"] = start_time.dt.day_name()

# Broad time-of-day groups.
# 00:00-05:59 Night, 06:00-11:59 Morning,
# 12:00-16:59 Afternoon, 17:00-23:59 Evening.
hour = start_time.dt.hour

df["TimeOfDay"] = np.select(
    [
        hour < 6,
        hour < 12,
        hour < 17,
    ],
    [
        "Night",
        "Morning",
        "Afternoon",
    ],
    default="Evening",
)

# Display example rows to verify the extracted values.
display(
    df[
        [
            "StartDatetime",
            "StartYear",
            "StartMonth",
            "StartDayOfMonth",
            "StartHour",
            "StartDayOfWeek",
            "StartDayName",
            "IsWeekend",
            "TimeOfDay",
        ]
    ].head(10)
)

,StartDatetime,StartYear,StartMonth,StartDayOfMonth,StartHour,StartDayOfWeek,StartDayName,IsWeekend,TimeOfDay
0,2022-09-15 20:54:02,2022,9,15,20,3,Thursday,0,Evening
1,2022-09-14 20:01:05,2022,9,14,20,2,Wednesday,0,Evening
2,2022-09-14 18:54:30,2022,9,14,18,2,Wednesday,0,Evening
3,2022-09-29 18:32:51,2022,9,29,18,3,Thursday,0,Evening
4,2022-09-25 19:30:15,2022,9,25,19,6,Sunday,1,Evening
5,2022-09-24 14:51:16,2022,9,24,14,5,Saturday,1,Afternoon
6,2022-08-11 18:28:42,2022,8,11,18,3,Thursday,0,Evening
7,2022-08-10 19:37:54,2022,8,10,19,2,Wednesday,0,Evening
8,2022-08-09 19:43:27,2022,8,9,19,1,Tuesday,0,Evening
9,2022-08-07 19:51:01,2022,8,7,19,6,Sunday,1,Evening


## Classification Target Engineering

The regression target remains the original continuous `Demand` variable.

For classification, we derive a separate target using the median demand of the cleaned dataset. This follows the threshold rule in the GUVI project brief.

The derived category is a target label, not an input feature. The original `Demand` column must never be included among the classification predictors.

In [4]:
# Calculate the median from the cleaned dataset instead
# of hardcoding the value observed before cleaning.
demand_threshold = df["Demand"].median()

df["DemandCategory"] = np.where(
    df["Demand"] >= demand_threshold,
    "High Demand",
    "Normal Demand",
)

print(f"Median demand threshold: {demand_threshold:.4f}")

print("\nClass counts:")
display(df["DemandCategory"].value_counts().to_frame("count"))

print("\nClass percentages:")
display(
    (
        df["DemandCategory"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_frame("percentage")
    )
)

Median demand threshold: 14.1000

Class counts:


,count
DemandCategory,
High Demand,36439
Normal Demand,36387



Class percentages:


,percentage
DemandCategory,
High Demand,50.04
Normal Demand,49.96


In [5]:
# These are candidate input features available at session start.
# We will use these lists consistently in the regression
# and classification notebooks.

CATEGORICAL_FEATURES = [
    "Location",
    "ChargerCompany",
    "ChargerType",
    "StartDayName",
    "TimeOfDay",
]

NUMERIC_FEATURES = [
    "StartMonth",
    "StartDayOfMonth",
    "StartHour",
    "IsWeekend",
]

MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Integer codes can represent categories, not quantities.
# Convert them to strings so preprocessing treats them as
# categories instead of implying that one code is "larger".
for col in ["ChargerCompany", "ChargerType"]:
    df[col] = df[col].astype("string")

print("Categorical predictors:", CATEGORICAL_FEATURES)
print("Numeric predictors:", NUMERIC_FEATURES)
print(f"Total candidate predictors: {len(MODEL_FEATURES)}")

display(df[MODEL_FEATURES].head())

Categorical predictors: ['Location', 'ChargerCompany', 'ChargerType', 'StartDayName', 'TimeOfDay']
Numeric predictors: ['StartMonth', 'StartDayOfMonth', 'StartHour', 'IsWeekend']
Total candidate predictors: 9


,Location,ChargerCompany,ChargerType,StartDayName,TimeOfDay,StartMonth,StartDayOfMonth,StartHour,IsWeekend
0,hotel,1,0,Thursday,Evening,9,15,20,0
1,hotel,1,0,Wednesday,Evening,9,14,20,0
2,hotel,1,0,Wednesday,Evening,9,14,18,0
3,hotel,1,0,Thursday,Evening,9,29,18,0
4,hotel,1,0,Sunday,Evening,9,25,19,1


In [6]:
# Confirm that the engineered columns exist and that key
# modelling targets remain usable.
required_columns = MODEL_FEATURES + [
    "Demand",
    "DemandCategory",
    "StartDatetime",
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values detected in the required model columns. "
        "Inspect them before continuing."
    )

# Save a separate model-ready CSV; do not overwrite cleaned data.
output_path = Path("../data/processed/ev_charging_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Engineered dataset shape: {df.shape}")
print(f"Saved engineered dataset to: {output_path.resolve()}")
print("Feature engineering checks passed.")

Engineered dataset shape: (72826, 24)
Saved engineered dataset to: E:\GUVI Projects\Ev_Charging_Demand_Project2\Ev_Charging_Demand_Prediction\data\processed\ev_charging_features.csv
Feature engineering checks passed.


# 7. Summary, Findings, and Conclusion

## 7.1 Objective Achieved

The objective of this notebook was to transform the cleaned EV charging dataset into a feature-rich dataset suitable for exploratory data analysis, regression, classification, and time-series analysis.

The feature-engineering process was completed successfully. Temporal features were extracted from charging start timestamps, a High/Normal Demand classification target was derived according to the GUVI project requirements, and an initial set of candidate predictors was established for supervised machine learning.

The original cleaned dataset was preserved. The engineered dataset was saved separately for use in subsequent notebooks.

## 7.2 Input Dataset

The input dataset was loaded from:

`data/processed/ev_charging_cleaned.csv`

It contained **72,826 records and 15 columns**, as produced by Notebook 01.

The cleaned data was loaded using pandas. The `StartDatetime` and `EndDatetime` columns were converted back into datetime objects because CSV files do not preserve pandas datetime data types.

The `ChargerCompany` and `ChargerType` columns were treated as categorical variables rather than continuous numerical measurements. Although their values are represented by integer-like codes, those codes identify categories rather than quantities.

## 7.3 Temporal Feature Engineering

The `StartDatetime` column was used to extract information about when each charging session began.

The following features were created:

| Feature | Description |
|---|---|
| `StartYear` | Calendar year of the session start |
| `StartMonth` | Month of the year, from 1 to 12 |
| `StartDayOfMonth` | Day of the month, from 1 to 31 |
| `StartHour` | Hour of the day, from 0 to 23 |
| `StartDayOfWeek` | Day-of-week index, where Monday = 0 and Sunday = 6 |
| `StartDayName` | Readable weekday name, such as Monday or Thursday |
| `IsWeekend` | 1 for Saturday or Sunday; otherwise 0 |
| `TimeOfDay` | Broad category derived from the start hour |

The `TimeOfDay` feature uses the following project-defined intervals:

| Time of day | Hour interval |
|---|---|
| Night | 00:00–05:59 |
| Morning | 06:00–11:59 |
| Afternoon | 12:00–16:59 |
| Evening | 17:00–23:59 |

These features make the temporal information easier to analyze and provide useful representations of charging-session start times.

For example, a session beginning on Saturday at 07:22 is categorized as a weekend morning session. The day name, weekend flag, hour, and time-of-day category can be used to investigate demand variations across different periods.

The day-of-week index and weekday name represent the same underlying calendar information in different forms. The initial model feature list uses `StartDayName` rather than `StartDayOfWeek`.

## 7.4 Classification Target Engineering

The continuous `Demand` variable remains the target for regression.

For classification, a new target column named `DemandCategory` was created using the threshold specified in the GUVI project brief:

- **High Demand:** `Demand >= median(Demand)`
- **Normal Demand:** `Demand < median(Demand)`

The median was recalculated using the cleaned dataset rather than hardcoded from the initial profiling results.

The resulting median demand threshold was **14.10**.

### Observed class distribution

| Category | Number of records | Percentage |
|---|---:|---:|
| High Demand | 36,439 | 50.04% |
| Normal Demand | 36,387 | 49.96% |
| **Total** | **72,826** | **100.00%** |

The two classes are almost perfectly balanced. Therefore, class-imbalance correction techniques such as SMOTE, ADASYN, and random oversampling are not necessary based on the observed class distribution.

The `DemandCategory` column is a classification target, not a predictor. The original `Demand` column must also be excluded from classification inputs because the category is derived directly from it.

## 7.5 Candidate Predictors for Supervised Learning

An initial set of nine candidate predictors was established for regression and classification.

### Categorical predictors

- `Location`
- `ChargerCompany`
- `ChargerType`
- `StartDayName`
- `TimeOfDay`

### Numeric predictors

- `StartMonth`
- `StartDayOfMonth`
- `StartHour`
- `IsWeekend`

The integer-like codes in `ChargerCompany` and `ChargerType` were converted to string data types so that the preprocessing pipelines can handle them as categorical variables.

These nine features provide information about charging infrastructure and session-start timing without directly incorporating the target or end-of-session measurements.

The candidate feature list is an initial modeling decision. It will be used consistently in the regression and classification notebooks unless subsequent validation provides a clear reason to revise it.

## 7.6 Avoiding Data Leakage

Data leakage occurs when a model uses information that would not legitimately be available when a prediction is made.

The primary prediction scenario in this project is to predict demand at the beginning of a charging session. Consequently, the candidate feature list excludes:

- `Demand`, because it is the regression target.
- `DemandCategory`, because it is the derived classification target.
- `Duration` and `DurationValidated`, because actual session duration is not known at session start.
- `EndDatetime`, `EndDay`, and `EndTime`, because these describe information from the end of the session.
- `UserID` and `ChargerID`, because these identifiers are not included in the initial predictive feature set.

The excluded columns remain available in the engineered dataset when required for other forms of analysis.

The time-series forecasting task will be handled separately. It will aggregate historical demand using appropriate timestamps rather than relying on the nine supervised-model predictors.

## 7.7 Validation and Output

The feature-engineering validation checks confirmed that the required candidate predictors, `Demand`, `DemandCategory`, and `StartDatetime` were present and contained no missing values.

The engineered dataset was successfully saved with the following results:

| Metric | Result |
|---|---:|
| Number of records | 72,826 |
| Number of columns | 24 |
| Candidate predictors | 9 |
| Classification target | `DemandCategory` |
| Regression target | `Demand` |
| Median demand threshold | 14.10 |

**Output file:** `data/processed/ev_charging_features.csv`

The engineered dataset retains the information required for EDA and subsequent modeling, including the original measurement fields and the newly created temporal and classification features.

## 7.8 Limitations and Considerations

- `StartYear` and `StartDayOfWeek` were engineered but are not included in the initial nine-feature predictor list.
- The `TimeOfDay` boundaries are project-defined categories and may not reflect every real-world charging pattern.
- The classification threshold is derived from the full cleaned dataset, following the GUVI brief. For strictly prospective deployment, the threshold and evaluation procedure should be designed using training data only.
- The class balance is based on the current dataset. The classification models must still be evaluated with appropriate metrics on held-out data.
- The engineered CSV may load integer-like category codes as numeric values. These columns should be converted to categorical types again when loading the dataset in subsequent notebooks.
- Feature engineering alone does not establish which predictors are important. Their predictive value will be investigated during modeling.

## 7.9 Conclusion and Next Steps

Feature engineering is complete. The cleaned dataset now contains temporal features, a derived High/Normal Demand target, and an initial set of nine candidate predictors for regression and classification.

The engineered dataset is available at `data/processed/ev_charging_features.csv`.

The next stage is Exploratory Data Analysis (EDA). It will investigate the distribution of demand, compare demand across locations and charger categories, study temporal patterns, and examine the relationship between validated duration and demand.

The EDA findings will provide evidence-based insights for the final project report and Streamlit dashboard. Subsequent modeling will use the documented candidate feature list and appropriate data splitting to avoid target leakage.